# Git Basics
**Topic:** Git & Version Control

In [ ]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Trace** the path a file takes from edited to staged to committed using `git add` and `git commit`
- **Explain** what `git status` and `git log` tell you about the state of your repository
- **Interpret** a commit message and explain what makes one useful versus useless

> **Tip:** Start by clicking the **Stage** button in the commit simulator widget with a file selected, then click **Commit** and watch the log grow before reading the explanation of what each step actually does.

---
## How we got here

In *01: What Is Version Control?* we established that a commit is a save point and that Git records a complete project history. Now we learn the mechanics: the exact commands that move a file from "I just edited this" to "this is saved forever in Git history." There are only five commands you need to know for your first month of Git, and this notebook covers them all.

---
## Why this matters for data science

Git is not optional in professional data science. A clean, descriptive commit history is evidence that you work methodically. It shows teammates what you changed and why. It is your audit trail when a stakeholder asks why the model changed between versions. The five commands in this notebook, `init`, `status`, `add`, `commit`, `log`, are the foundation everything else builds on.

---
## Try it yourself

In [ ]:
# Widget: A commit simulator with three panels
# Left panel: "Working Directory" — a list of file cards with checkboxes
#   Pre-loaded files: clean.py (modified), notebook.ipynb (modified), README.md (untracked)
# Middle panel: "Staging Area" — files move here when the Stage button is clicked
# Right panel: "Git Log" — grows as commits are made
# Controls:
#   A text input for the commit message
#   A "Stage selected files" button (moves checked files from Working Dir to Staging)
#   A "Commit" button (creates a commit entry in the log panel, clears the staging area)
#   An "Unstage" button (moves files back from staging to working directory)
# Each commit in the log shows: hash (fake 7-char hex), author, timestamp, message
# Student should notice: only staged files go into the commit;
#   untracked files (never added) do not appear in git status as staged even if modified

---
## What's happening?

Git tracks changes through three zones. Understanding the zones makes every Git command make sense.

```
 WORKING DIRECTORY          STAGING AREA            REPOSITORY
 (files on disk)            (index)                 (.git/ folder)

  clean.py  [modified]  ──git add──►  clean.py     ──git commit──►  commit a3f7c1
  notes.txt [untracked]                                               │ "Add data cleaning step"
  README.md [unchanged]                                               │
                                                                     commit 9b2d4e
                                                                      │ "Initial commit"
```

| Command | What it does | When to use it |
|---------|-------------|---------------|
| `git init` | Create a new repository in the current directory | Once, at the start of every new project |
| `git status` | Show which files are modified, staged, or untracked | Before every commit to see what you are about to save |
| `git add <file>` | Move a file from working directory to staging area | After every edit you want to include in the next commit |
| `git add .` | Stage all modified and untracked files | Quick staging when all changes belong in one commit |
| `git commit -m "message"` | Save staged changes as a permanent snapshot | After staging; write a message that explains *why* you changed something |
| `git log` | Show the commit history | To see what you (and teammates) have committed recently |
| `git diff` | Show what has changed since the last commit | Before staging, to review your own changes |

### What makes a good commit message?

A good commit message completes this sentence: "If applied, this commit will ___."

| Bad | Good | Why |
|-----|------|-----|
| `"fix"` | `"Fix off-by-one error in feature engineering window calculation"` | Says *what* was fixed |
| `"update notebook"` | `"Add EDA for customer age distribution"` | Says *what* was added |
| `"WIP"` | `"Draft feature selection pipeline (variance threshold)"` | Distinguishes draft from final |

Return to the simulator and practice making three commits with progressively more descriptive messages.

---
## Real-world example: A week of commits on a churn prediction project

The chart below shows a realistic week of Git activity on a data science project. Each row is one commit; the colored segments show which types of files were changed. This is the kind of history a hiring manager looks at when they review your GitHub profile.

Notice:

- **Notice:** Good commit history is granular: multiple small, focused commits per day rather than one enormous commit at the end of the week, which makes each change easy to find and revert independently
- **Notice:** The commit messages describe the *purpose* of the change, not just the file name; "Add log transform to income feature" tells a future reader what they need to know without opening the diff
- **Notice:** The pattern of changes (data → notebook → model → evaluation) reflects the natural order of a data science workflow, which means the history also tells the story of the project

> **Discussion question:** A teammate's commit history for the week is a single commit called "all the stuff from this week." What specific problems does this cause if you later need to revert just the model change without touching the data cleaning changes?

In [ ]:
# ── Simulated week of Git commits on a data science project ────────────────────
commits_data = [
    ("Mon 09:12", "chore: init project structure and environment.yml",      2, 0, 0),
    ("Mon 14:33", "feat: add raw customer data loader with schema checks",   1, 3, 0),
    ("Tue 10:05", "feat: add EDA notebook for churn distribution",          0, 1, 0),
    ("Tue 16:41", "fix: correct null handling in tenure_days column",       1, 1, 0),
    ("Wed 11:20", "feat: add log transform to income feature",              1, 0, 0),
    ("Wed 15:08", "feat: draft logistic regression baseline (AUC 0.72)",    0, 1, 1),
    ("Thu 10:45", "refactor: extract feature pipeline into src/features.py",3, 0, 0),
    ("Thu 14:22", "feat: add SMOTE oversampling for class imbalance",       1, 0, 1),
    ("Fri 09:55", "test: add unit tests for clean_tenure() function",       2, 0, 0),
    ("Fri 15:30", "docs: update README with project setup instructions",    0, 0, 0),
]
times   = [c[0] for c in commits_data]
msgs    = [c[1] for c in commits_data]
py_ch   = [c[2] for c in commits_data]
nb_ch   = [c[3] for c in commits_data]
mod_ch  = [c[4] for c in commits_data]

fig = go.Figure()
for vals, name, color in [
    (py_ch,  "Python files changed", PALETTE["primary"]),
    (nb_ch,  "Notebooks changed",    PALETTE["secondary"]),
    (mod_ch, "Model outputs changed",PALETTE["accent"]),
]:
    fig.add_trace(go.Bar(
        x=vals, y=times,
        orientation="h",
        name=name, marker_color=color,
        customdata=msgs,
        hovertemplate="%{customdata}<extra></extra>",
    ))
layout = base_layout(
    title="One Week of Git Commits — Churn Prediction Project",
    xaxis_title="Files Changed",
    yaxis_title="",
)
layout.update(barmode="stack", yaxis=dict(autorange="reversed"))
fig.update_layout(layout)
fig.show()

### Essential git commands reference

| Command | What it does | When to use it |
|---------|-------------|---------------|
| `git init` | Initialize a new repository | Start of every project |
| `git status` | Show modified, staged, and untracked files | Before every `add` and `commit` |
| `git add <file>` | Stage a specific file | When you want precise control over what goes into a commit |
| `git add .` | Stage all changes in the current directory | When all changes belong in one commit |
| `git commit -m "msg"` | Commit staged changes with a message | After staging; use a descriptive message |
| `git log --oneline` | Show compact commit history | Quick review of recent commits |
| `git diff` | Show unstaged changes | Before adding, to review your edits |
| `git diff --staged` | Show staged changes | After adding, to verify what will be committed |
| `git show <hash>` | Show the full diff for one commit | When investigating a specific change |
| `git restore <file>` | Discard unstaged changes in a file | To undo edits you do not want to keep |
| `git restore --staged <file>` | Unstage a file | If you added something by mistake |

---
## Key takeaway

> **Git's core workflow is three steps: edit files in the working directory, stage the ones you want to save with `git add`, and permanently record them with `git commit`; the log is your undo history for everything that has been committed.**

---
*Next up: Branching & Merging — how Git lets you experiment in parallel without risking the working version of your project*